In [1]:
# Phase 5 - Guarded Analysis
# The 12 business questions, with guardrails enforced in the code.

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import stats_helpers as sh

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

rows     = pd.read_csv(ROOT / "data/processed/birds_features.csv", encoding="utf-8")
sessions = pd.read_csv(ROOT / "data/processed/sessions.csv", encoding="utf-8")
species  = pd.read_csv(ROOT / "data/processed/species_profile.csv", encoding="utf-8")
parks    = pd.read_csv(ROOT / "data/reference/park_coordinates.csv", encoding="utf-8")

# ---------------------------------------------------------------
# THE FOUR GUARDRAILS - every answer below obeys these.
#
# G1  Never compare raw counts across habitats. Use per-session rates.
# G2  Habitat comparisons happen WITHIN a park, using only the four
#     parks where both habitats were surveyed.
# G3  Use per-session, never per-plot (forest 2.0 visits, grassland 2.96).
# G4  Charts and maps encode adjusted rates, never raw totals.
# ---------------------------------------------------------------

SHARED_PARKS = ["ANTI", "HAFE", "MANA", "MONO"]

# G2 in one line: this is the only slice valid for habitat comparison.
shared = sessions[sessions.is_shared_park]

print(f"all sessions            : {len(sessions):,}")
print(f"sessions in shared parks: {len(shared):,}  <- G2 applies to these")
print(f"shared parks            : {SHARED_PARKS}")

all sessions            : 1,408
sessions in shared parks: 737  <- G2 applies to these
shared parks            : ['ANTI', 'HAFE', 'MANA', 'MONO']


In [2]:
# Q1 - At-risk species rate by habitat, WITHIN each shared park (G1 + G2)

print("Q1  AT-RISK SPECIES RATE BY HABITAT")
print("=" * 58)

# A rate, not a count: at-risk sightings divided by all sightings.
by_park = (
    shared.groupby(["Admin_Unit_Code", "habitat"])
    .apply(lambda g: g.at_risk_sightings.sum() / g.sightings_per_session.sum() * 100,
           include_groups=False)
    .unstack()
    .round(2)
)
by_park["forest_higher"] = by_park.Forest > by_park.Grassland
print(by_park.to_string())
print()

f = shared[shared.habitat == "Forest"]
g = shared[shared.habitat == "Grassland"]
f_rate = f.at_risk_sightings.sum() / f.sightings_per_session.sum() * 100
g_rate = g.at_risk_sightings.sum() / g.sightings_per_session.sum() * 100

print(f"Pooled across the 4 shared parks:")
print(f"  Forest    {f_rate:.2f}%")
print(f"  Grassland {g_rate:.2f}%")
print(f"  Ratio     {f_rate / g_rate:.1f}x")
print()

# Is it statistically real? Compare the per-session at-risk % between habitats.
a = f.pct_at_risk_per_session.values
b = g.pct_at_risk_per_session.values
_, p = sh.mannwhitneyu(a, b)
print(f"Mann-Whitney p = {p:.3g}   {'SIGNIFICANT' if p < 0.05 else 'not significant'}")

Q1  AT-RISK SPECIES RATE BY HABITAT
habitat          Forest  Grassland  forest_higher
Admin_Unit_Code                                  
ANTI               2.70       0.42           True
HAFE               5.92       0.00           True
MANA               2.37       1.32           True
MONO               3.51       0.37           True

Pooled across the 4 shared parks:
  Forest    3.65%
  Grassland 0.59%
  Ratio     6.2x

Mann-Whitney p = 1.66e-18   SIGNIFICANT


In [3]:
# Q2 - Species richness per session by habitat
# Computed TWICE, deliberately: the wrong way, then the right way.

print("Q2  SPECIES RICHNESS PER SESSION")
print("=" * 58)

# --- THE WRONG WAY: pool all 11 parks (violates G2) -------------
a_all = sessions[sessions.habitat == "Forest"].species_per_session.values
b_all = sessions[sessions.habitat == "Grassland"].species_per_session.values
_, p_all = sh.mannwhitneyu(a_all, b_all)

print("Pooled across ALL 11 parks  (violates G2 - do not report this):")
print(f"  Forest    {a_all.mean():.2f} species per session  (n={len(a_all):,})")
print(f"  Grassland {b_all.mean():.2f} species per session  (n={len(b_all):,})")
print(f"  p = {p_all:.6f}  ->  {'SIGNIFICANT' if p_all < 0.05 else 'not significant'}")
print()

# --- THE RIGHT WAY: within the shared parks only (obeys G2) -----
a_sh = shared[shared.habitat == "Forest"].species_per_session.values
b_sh = shared[shared.habitat == "Grassland"].species_per_session.values
_, p_sh = sh.mannwhitneyu(a_sh, b_sh)

print("Within the 4 shared parks  (obeys G2 - this is the answer):")
print(f"  Forest    {a_sh.mean():.2f} species per session  (n={len(a_sh):,})")
print(f"  Grassland {b_sh.mean():.2f} species per session  (n={len(b_sh):,})")
print(f"  p = {p_sh:.6f}  ->  {'SIGNIFICANT' if p_sh < 0.05 else 'not significant'}")
print()

print("Per park, to show it is not one park driving anything:")
print(shared.groupby(["Admin_Unit_Code", "habitat"])
      .species_per_session.mean().unstack().round(2).to_string())

Q2  SPECIES RICHNESS PER SESSION
Pooled across ALL 11 parks  (violates G2 - do not report this):
  Forest    8.61 species per session  (n=814)
  Grassland 9.15 species per session  (n=594)
  p = 0.000450  ->  SIGNIFICANT

Within the 4 shared parks  (obeys G2 - this is the answer):
  Forest    9.10 species per session  (n=143)
  Grassland 9.15 species per session  (n=594)
  p = 0.685483  ->  not significant

Per park, to show it is not one park driving anything:
habitat          Forest  Grassland
Admin_Unit_Code                   
ANTI              10.65       9.98
HAFE               8.57       9.22
MANA               8.19       8.56
MONO               9.90       8.49


In [4]:
# Q3 - Top parks and plots by species per session (G1, G4)
# Ranked by an effort-adjusted rate, never by raw totals.

print("Q3  BIODIVERSITY HOTSPOTS")
print("=" * 58)

park_rank = (
    sessions.groupby("Admin_Unit_Code")
    .agg(sessions_run=("species_per_session", "size"),
         species_per_visit=("species_per_session", "mean"),
         distinct_species=("Admin_Unit_Code", "size"))
    .drop(columns="distinct_species")
)
park_rank["distinct_species"] = rows.groupby("Admin_Unit_Code").Scientific_Name.nunique()
park_rank = park_rank.sort_values("species_per_visit", ascending=False).round(2)
print("PARKS RANKED BY SPECIES PER SESSION (the correct ranking)")
print(park_rank.to_string())
print()

print("THE SAME PARKS RANKED BY RAW SPECIES COUNT (the misleading one)")
print(park_rank.sort_values("distinct_species", ascending=False)
      .head(5).to_string())
print()

top_plots = (
    sessions[sessions.groupby("Plot_Name").species_per_session.transform("size") >= 2]
    .groupby(["Plot_Name", "Admin_Unit_Code", "habitat"])
    .species_per_session.agg(["mean", "size"])
    .sort_values("mean", ascending=False).head(10).round(2)
)
print("TOP 10 PLOTS (visited at least twice, so the average means something)")
print(top_plots.to_string())

Q3  BIODIVERSITY HOTSPOTS
PARKS RANKED BY SPECIES PER SESSION (the correct ranking)
                 sessions_run  species_per_visit  distinct_species
Admin_Unit_Code                                                   
ANTI                      278              10.04                81
CHOH                      180               9.95                80
NACE                       58               9.52                66
WOTR                       12               8.83                27
HAFE                       49               8.69                55
MONO                      232               8.67               100
MANA                      178               8.46                81
ROCR                       28               8.25                45
PRWI                      264               7.85                54
GWMP                       40               7.45                49
CATO                       89               7.33                46

THE SAME PARKS RANKED BY RAW SPECIES COUNT (

In [5]:
# Q4 - Habitat preference per species (G2: shared parks only)

print("Q4  HABITAT SPECIALISTS")
print("=" * 58)

well = species[species.well_sampled]
print(f"Species with at least 20 sightings in the shared parks: {len(well)}")
print()

print(species.specialist_class.value_counts().to_string())
print()

print("GRASSLAND SPECIALISTS (>90% of sightings in grassland)")
gs = well[well.specialist_class == "Grassland specialist"].sort_values(
    "grassland_share_pct", ascending=False)
print(gs[["Common_Name", "forest_sightings", "grassland_sightings",
          "grassland_share_pct", "is_at_risk"]].to_string(index=False))
print()

print("FOREST SPECIALISTS (<10% of sightings in grassland)")
fs = well[well.specialist_class == "Forest specialist"]
print(fs[["Common_Name", "forest_sightings", "grassland_sightings",
          "grassland_share_pct", "is_at_risk"]].to_string(index=False))

Q4  HABITAT SPECIALISTS
Species with at least 20 sightings in the shared parks: 49

specialist_class
Insufficient data       66
Generalist              30
Grassland specialist    18
Forest specialist        1

GRASSLAND SPECIALISTS (>90% of sightings in grassland)
                  Common_Name  forest_sightings  grassland_sightings  grassland_share_pct  is_at_risk
                 Tree Swallow                 0                   49                100.0       False
           American Goldfinch                 0                  310                100.0       False
            European Starling                 1                   81                 98.8       False
          Grasshopper Sparrow                 5                  346                 98.6       False
                 Barn Swallow                 2                  130                 98.5       False
             Eastern Kingbird                 1                   50                 98.0       False
                 Song

In [6]:
# Q5 - Which watchlist species occur where (Objectives 1 and 5)

print("Q5  AT-RISK SPECIES: WHICH, AND WHERE")
print("=" * 58)

at_risk = rows[rows.is_at_risk]

print(f"{at_risk.Scientific_Name.nunique()} at-risk species, "
      f"{len(at_risk):,} sightings\n")

profile = (
    at_risk.groupby(["Common_Name", "habitat"]).size().unstack(fill_value=0)
)
for h in ("Forest", "Grassland"):
    if h not in profile.columns:
        profile[h] = 0
profile["total"] = profile.Forest + profile.Grassland
profile["parks"] = at_risk.groupby("Common_Name").Admin_Unit_Code.nunique()
print(profile.sort_values("total", ascending=False).to_string())
print()

print("AT-RISK RATE BY PARK (all 11 parks, effort-adjusted)")
by_park = (
    rows.groupby("Admin_Unit_Code")
    .agg(at_risk_pct=("is_at_risk", lambda s: s.mean() * 100),
         sightings=("is_at_risk", "size"))
    .round(2).sort_values("at_risk_pct", ascending=False)
)
by_park["at_risk_species"] = at_risk.groupby("Admin_Unit_Code").Scientific_Name.nunique()
print(by_park.fillna(0).to_string())

Q5  AT-RISK SPECIES: WHICH, AND WHERE
8 at-risk species, 378 sightings

habitat                Forest  Grassland  total  parks
Common_Name                                           
Wood Thrush               290         19    309     10
Worm-eating Warbler        31          0     31      5
Prairie Warbler             7         18     25      3
Cerulean Warbler            7          0      7      3
Willow Flycatcher           0          2      2      2
Kentucky Warbler            1          1      2      2
Blue-winged Warbler         1          0      1      1
Red-headed Woodpecker       1          0      1      1

AT-RISK RATE BY PARK (all 11 parks, effort-adjusted)
                 at_risk_pct  sightings  at_risk_species
Admin_Unit_Code                                         
CATO                    8.45        805              3.0
PRWI                    5.60       2463              4.0
ROCR                    4.84        289              1.0
HAFE                    4.72        530

In [7]:
# Q6 - Monthly species richness by habitat
# Q7 - Richness by time-of-day band

print("Q6  MONTHLY RICHNESS")
print("=" * 58)

monthly = (sessions.groupby(["month_name", "habitat"])
           .agg(sessions_run=("species_per_session", "size"),
                species_per_session=("species_per_session", "mean"))
           .round(2).unstack())
print(monthly.reindex(["May", "June", "July"]).to_string())
print()
print("CAVEAT: forest sampling is uneven across months (June ~2x May/July),")
print("so monthly forest figures are descriptive only, not a trend.")
print()

print("Q7  TIME-OF-DAY BAND")
print("=" * 58)

band = (sessions.groupby(["time_band", "habitat"])
        .agg(sessions_run=("species_per_session", "size"),
             species_per_session=("species_per_session", "mean"))
        .round(2).unstack())
order = ["Early (5-6am)", "Mid (7-8am)", "Late (9-10am)"]
print(band.reindex([b for b in order if b in band.index]).to_string())
print()

# Is early significantly better than late, within each habitat?
for hab in ["Forest", "Grassland"]:
    s = sessions[sessions.habitat == hab]
    early = s[s.time_band == "Early (5-6am)"].species_per_session.values
    late = s[s.time_band == "Late (9-10am)"].species_per_session.values
    if len(early) and len(late):
        _, p = sh.mannwhitneyu(early, late)
        diff = early.mean() - late.mean()
        print(f"  {hab:<10} early {early.mean():.2f} vs late {late.mean():.2f}  "
              f"(+{diff:.2f})  p={p:.4g}  "
              f"{'SIGNIFICANT' if p < 0.05 else 'not significant'}")

Q6  MONTHLY RICHNESS
           sessions_run           species_per_session          
habitat          Forest Grassland              Forest Grassland
month_name                                                     
May                 212       198                9.22      9.85
June                391       194                8.40      9.15
July                211       202                8.37      8.45

CAVEAT: forest sampling is uneven across months (June ~2x May/July),
so monthly forest figures are descriptive only, not a trend.

Q7  TIME-OF-DAY BAND
              sessions_run           species_per_session          
habitat             Forest Grassland              Forest Grassland
time_band                                                         
Early (5-6am)          270       179                8.65      9.70
Mid (7-8am)            406       254                8.66      9.32
Late (9-10am)          138       161                8.36      8.25

  Forest     early 8.65 vs late 8.36  (

In [8]:
# Q8 - Richness against temperature and humidity bands
# Q9 - Richness by sky, wind and disturbance

print("Q8  TEMPERATURE AND HUMIDITY")
print("=" * 58)

temp_order = ["<15C", "15-20C", "20-25C", "25-30C", ">30C"]
t = (sessions.groupby("temp_band")
     .agg(sessions_run=("species_per_session", "size"),
          species_per_session=("species_per_session", "mean")).round(2))
print(t.reindex([b for b in temp_order if b in t.index]).to_string())
print()

hum_order = ["<40%", "40-60%", "60-80%", ">80%"]
h = (sessions.groupby("humidity_band")
     .agg(sessions_run=("species_per_session", "size"),
          species_per_session=("species_per_session", "mean")).round(2))
print(h.reindex([b for b in hum_order if b in h.index]).to_string())
print()

print("Q9  SKY, WIND AND DISTURBANCE")
print("=" * 58)

for col in ["Sky", "Wind", "Disturbance"]:
    d = (sessions.groupby(col)
         .agg(sessions_run=("species_per_session", "size"),
              species_per_session=("species_per_session", "mean"))
         .round(2).sort_values("species_per_session", ascending=False))
    print(f"\n{col.upper()}")
    print(d.to_string())

# Is the disturbance effect statistically real at the serious end?
none = sessions[sessions.Disturbance == "No effect on count"].species_per_session.values
bad = sessions[sessions.Disturbance == "Serious effect on count"].species_per_session.values
_, p = sh.mannwhitneyu(none, bad)
print(f"\n  none ({none.mean():.2f}) vs serious ({bad.mean():.2f})  p={p:.3g}  "
      f"{'SIGNIFICANT' if p < 0.05 else 'not significant'}")

Q8  TEMPERATURE AND HUMIDITY
           sessions_run  species_per_session
temp_band                                   
<15C                 50                 9.10
15-20C              314                 9.61
20-25C              658                 8.91
25-30C              313                 8.04
>30C                 73                 7.97

               sessions_run  species_per_session
humidity_band                                   
<40%                     12                11.50
40-60%                  170                 8.67
60-80%                  704                 8.68
>80%                    522                 9.03

Q9  SKY, WIND AND DISTURBANCE

SKY
                     sessions_run  species_per_session
Sky                                                   
Partly Cloudy                 542                 9.20
Cloudy/Overcast               270                 8.76
Clear or Few Clouds           496                 8.75
Fog                            61                 

In [9]:
# Q10 - Observer counts and mean richness (bias disclosure)
# Q11 - Species recorded in one habitat only
# Q12 - Survey effort by park and habitat (coverage transparency)

print("Q10  OBSERVER DISCLOSURE")
print("=" * 58)
obs = (sessions.groupby("Observer")
       .agg(sessions_run=("species_per_session", "size"),
            species_per_session=("species_per_session", "mean"),
            spread=("species_per_session", "std")).round(2))
print(obs.to_string())
print("\nBalance check - sessions by habitat:")
print(pd.crosstab(sessions.habitat, sessions.Observer).to_string())
print()

print("Q11  SPECIES IN ONE HABITAT ONLY (shared parks, G2)")
print("=" * 58)
sh_rows = rows[rows.is_shared_park]
f_sp = set(sh_rows[sh_rows.habitat == "Forest"].Scientific_Name)
g_sp = set(sh_rows[sh_rows.habitat == "Grassland"].Scientific_Name)
print(f"  forest only    : {len(f_sp - g_sp)}")
print(f"  grassland only : {len(g_sp - f_sp)}")
print(f"  both           : {len(f_sp & g_sp)}")
print("\n  (Computed on shared parks only. Across all 11 parks the numbers")
print("   are 20/19/88, but those are confounded - a species can look")
print("   'forest only' simply because grassland was never surveyed in its park.)")
print()

print("Q12  SURVEY EFFORT - THE COVERAGE TABLE")
print("=" * 58)
eff = (sessions.groupby(["Admin_Unit_Code", "habitat"]).size()
       .unstack(fill_value=0))
eff["total"] = eff.sum(axis=1)
eff["both_habitats"] = (eff.Forest > 0) & (eff.Grassland > 0)
print(eff.sort_values("total", ascending=False).to_string())
print(f"\n  Parks with both habitats: {eff.both_habitats.sum()} of {len(eff)}")
print(f"  Sessions usable for habitat comparison: {len(sessions[sessions.is_shared_park]):,} of {len(sessions):,}")

Q10  OBSERVER DISCLOSURE
                  sessions_run  species_per_session  spread
Observer                                                   
Brian Swimelar             461                 7.27    2.31
Elizabeth Oswald           490                 9.96    3.26
Kimberly Serno             457                 9.21    2.74

Balance check - sessions by habitat:
Observer   Brian Swimelar  Elizabeth Oswald  Kimberly Serno
habitat                                                    
Forest                264               291             259
Grassland             197               199             198

Q11  SPECIES IN ONE HABITAT ONLY (shared parks, G2)
  forest only    : 8
  grassland only : 37
  both           : 70

  (Computed on shared parks only. Across all 11 parks the numbers
   are 20/19/88, but those are confounded - a species can look
   'forest only' simply because grassland was never surveyed in its park.)

Q12  SURVEY EFFORT - THE COVERAGE TABLE
habitat          Forest  Grasslan